In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils import dados, pivot, salvar

import numpy as np

In [2]:
# Dados

dados_brutos, metadados, variaveis = dados(r"data\input\dados_brutos.xlsx")

dados_limpos, metadados, variaveis = dados(r"data\input\dados_limpos.xlsx")

imput_mean_limpos, metadados, variaveis = dados(r"data\output\imput_mean_limpos.xlsx")

imput_median_limpos, metadados, variaveis = dados(r"data\output\imput_median_limpos.xlsx")

imput_knn_limpos, metadados, variaveis = dados(r"data\output\imput_knn_limpos.xlsx")

imput_mean_zscore, metadados, variaveis = dados(r"data\input\imput_mean_zscore.xlsx")

imput_median_zscore, metadados, variaveis = dados(r"data\input\imput_median_zscore.xlsx")

imput_knn_zscore, metadados, variaveis = dados(r"data\input\imput_knn_zscore.xlsx")

imput_mean_iqr, metadados, variaveis = dados(r"data\input\imput_mean_iqr.xlsx")

imput_median_iqr, metadados, variaveis = dados(r"data\input\imput_median_iqr.xlsx")

imput_knn_iqr, metadados, variaveis = dados(r"data\input\imput_knn_iqr.xlsx")


# Preparação para cálculos das métricas

In [ ]:
def preparar_para_metricas(df_verdadeiro, df_predito, metadados):
    """
    Prepara dois DataFrames para cálculo de métricas.
    
    Remove metadados e garante que ambos tenham as mesmas colunas numéricas.
    
    Parameters
    ----------
    df_verdadeiro : pd.DataFrame
        DataFrame com valores verdadeiros.
    df_predito : pd.DataFrame
        DataFrame com valores preditos/imputados.
    metadados : list
        Lista de colunas de metadados a ignorar.
    
    Returns
    -------
    df_verdadeiro_prep : pd.DataFrame
        DataFrame preparado (apenas colunas numéricas comuns, sem metadados).
    df_predito_prep : pd.DataFrame
        DataFrame preparado (apenas colunas numéricas comuns, sem metadados).
    colunas_usadas : list
        Lista das colunas que serão usadas nas métricas.
    """
    # Eu presumo que os dados_verdadeiros serão uma das bases de dados produzidas até
    # o momento, e que os dados_preditos terão alguma coisa a ver com os modelos de ML.

    # Busca a presença de NaN no DataFrame
    if df_verdadeiro.columns[df_verdadeiro.isna().any()].array.size != 0:
    # Remove linhas com NaN, caso elas existam
        df_verdadeiro = df_verdadeiro.dropna().copy()
    # Mantém df_verdadeiro, caso ele já não apresente NaN
    else:
        pass

    # Encontra colunas comuns (intersecção)
    colunas_comuns = set(df_verdadeiro.columns) & set(df_predito.columns)
    
    # Remove metadados. Importante, pois os cálculos de métricas só funcionam em colunas numéricas.
    # Exemplo: Nas colunas de datas (dtype=datetime), os métodos sklearn retornam TypeError
    colunas_usadas = [col for col in colunas_comuns if col not in metadados]
    
    df_verdadeiro_prep = df_verdadeiro[colunas_usadas]
    df_predito_prep = df_predito[colunas_usadas]
    
    # Usa apenas as colunas que existem em ambos 
    colunas_usadas = list(df_verdadeiro_prep.columns)
    df_predito_prep = df_predito_prep[colunas_usadas]
    
    print(f"Colunas usadas para métricas: {colunas_usadas}")
    print(f"Total de colunas: {len(colunas_usadas)}")
    
    return df_verdadeiro_prep, df_predito_prep, colunas_usadas

# Arquitetura Simplificada para Avaliação de Imputações

In [ ]:
def simular_dados_faltantes(df, cenario="a", pct_remover=0.05, metadados=None, seed=42):
    """
    Versão simplificada: cria dataset com NaN simulado.
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataset base (será removido NaN primeiro).
    cenario : str, default "a"
        "a" = Remove apenas das colunas que tinham NaN original
        "b" = Remove de TODAS as colunas (exceto metadados)
    pct_remover : float
        Percentual de valores a remover (ex: 0.05 para 5%).
    metadados : list, optional
        Colunas a ignorar.
    seed : int
        Seed para reprodutibilidade.
    
    Returns
    -------
    df_referencia : pd.DataFrame
        Dataset sem NaN (referência verdadeira).
    df_simulado : pd.DataFrame
        Dataset com NaN simulado.
    """
    
    np.random.seed(seed)
    if metadados is None:
        metadados = []
    
    # Remove todas as linhas com NaN
    df_ref = df.dropna().copy()
    
    # Identifica colunas que originalmente tinham NaN
    cols_com_nan = df.columns[df.isna().any()].tolist()
    cols_com_nan = [col for col in cols_com_nan if col not in metadados]
    
    # Todas as colunas numéricas (exceto metadados)
    todas_cols = [col for col in df_ref.columns if col not in metadados]
    
    # Cria dataset simulado
    df_sim = df_ref.copy()
    n_remover = int(len(df_sim) * pct_remover)
    
    # Escolhe cenário
    if cenario.lower() == "a":
        colunas_alvo = cols_com_nan
    elif cenario.lower() == "b":
        colunas_alvo = todas_cols
    else:
        raise ValueError("cenario deve ser 'a' ou 'b'")
    
    # Remove valores aleatoriamente
    for col in colunas_alvo:
        indices = np.random.choice(df_sim.index, n_remover, replace=False)
        df_sim.loc[indices, col] = np.nan
    
    print(f"Cenário {cenario.upper()}: {len(colunas_alvo)} colunas, {pct_remover*100}% removido")
    
    return df_ref, df_sim

In [ ]:
def avaliar_imputacao(df_verdadeiro, df_imputado, metrica="rmse", cenario="a", metadados=None):
    """
    Interface simples para avaliar qualidade de imputação.
    
    Compara dataset verdadeiro (referência) com dataset imputado.
    
    Parameters
    ----------
    df_verdadeiro : pd.DataFrame
        Dataset verdadeiro (sem manipulações).
    df_imputado : pd.DataFrame
        Dataset que recebeu alguma transformação (imputação, etc).
    metrica : str, default "rmse"
        Métrica a calcular: "rmse", "mae", "r2".
    cenario : str, default "a"
        "a" = avalia apenas colunas que tinham NaN original
        "b" = avalia TODAS as colunas
    metadados : list, optional
        Colunas de metadados a ignorar.
    
    Returns
    -------
    resultado : float
        Valor da métrica calculada.
    """
    
    if metadados is None:
        metadados = []
    
    # Preparar dados
    df_verd_prep, df_imput_prep, colunas = preparar_para_metricas(
        df_verdadeiro, df_imputado, metadados
    )
    
    # Calcular métrica
    if metrica.lower() == "rmse":
        from sklearn.metrics import root_mean_squared_error
        resultado = root_mean_squared_error(df_verd_prep, df_imput_prep)
    
    elif metrica.lower() == "mae":
        from sklearn.metrics import mean_absolute_error
        resultado = mean_absolute_error(df_verd_prep, df_imput_prep)
    
    elif metrica.lower() == "r2":
        from sklearn.metrics import r2_score
        resultado = r2_score(df_verd_prep, df_imput_prep)
    
    else:
        raise ValueError(f"Métrica desconhecida: {metrica}")
    
    print(f"{metrica.upper()}: {resultado:.6f}")
    return resultado

In [6]:
def criar_validacao_cruzada(df, pct_remover=0.05, metadados=None, seed=42):
    """
    Cria datasets com missing data simulado para validação cruzada.
    
    Gera 2 cenários:
    - Cenário A: Remove apenas das colunas que tinham NaN original
    - Cenário B: Remove de TODAS as colunas (exceto metadados)
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataset com alguns NaN (não será modificado).
    pct_remover : float
        Percentual de valores a remover (ex: 0.05 para 5%). Default 0.05.
    metadados : list, optional
        Colunas a não considerar. Default None.
    seed : int
        Seed para reprodutibilidade. Default 42.
    
    Returns
    -------
    df_nan : pd.DataFrame
        Dataset sem nenhum NaN (referência verdadeira).
    dados_cenario_a : pd.DataFrame
        Dataset com NaN simulado apenas nas colunas que tinham NaN original.
    dados_cenario_b : pd.DataFrame
        Dataset com NaN simulado em TODAS as colunas.
    colunas_com_nan_original : list
        Quais colunas tinham NaN no dataset original.
    """
    
    np.random.seed(seed)
    
    if metadados is None:
        metadados = []
    
    # Cria uma versão de df sem NaN
    df_nan = df.dropna().copy()
    
    # Identifica colunas que originalmente tinham NaN 
    colunas_com_nan_original = df.columns[df.isna().any()].tolist()
    colunas_com_nan_original = [col for col in colunas_com_nan_original if col not in metadados]
    
    # Colunas para simulação (excluindo metadados)
    todas_colunas_numericas = [col for col in df_nan.columns if col not in metadados]
    
    # Cenário A: Remove NaN apenas das colunas que tinham NaN
    dados_cenario_a = df_nan.copy()
    n_valores_remover = int(len(dados_cenario_a) * pct_remover)
    for col in colunas_com_nan_original:
        indices = np.random.choice(dados_cenario_a.index, n_valores_remover, replace=False)
        dados_cenario_a.loc[indices, col] = np.nan
    
    # Cenário B: Remove NaN de TODAS as colunas
    dados_cenario_b = df_nan.copy()
    for col in todas_colunas_numericas:
        indices = np.random.choice(dados_cenario_b.index, n_valores_remover, replace=False)
        dados_cenario_b.loc[indices, col] = np.nan
    
    print(f"Dataset completo: {df_nan.shape}")
    print(f"Colunas com NaN original: {colunas_com_nan_original}")
    print(f"Valores removidos simulados: {pct_remover*100}% ({n_valores_remover} por coluna)")
    
    return dados_cenario_a, dados_cenario_b

In [ ]:
def prep_validacao(df, pct_remover, seed=0.42):
    from utils import separar_colunas
    metadados, variaveis = separar_colunas(df)

    # Busca a presença de NaN no DataFrame
    if df.columns[df.isna().any()].array.size != 0:
    # Remove linhas com NaN, caso elas existam
        df = df.dropna().copy()
    # Mantém df, caso ele já não apresente NaN
    else:
        pass

    

In [ ]:
def avaliacao(df, modelo, metrica):

    # Listas de métricas e modelos disponíveis
    lista_modelos = ['media', 'mediana', 'knn']
    lista_metricas = ['RMSE', 'MAE', 'bias', 'r2']

    # Busca a presença de NaN no DataFrame
    if df.columns[df.isna().any()].array.size != 0:
    # Remove linhas com NaN, caso elas existam
        df = df.dropna().copy()
    # Mantém df, caso ele já não apresente NaN
    else:
        df = df.copy()   

    # Armazena as listas de metadados e variáveis de df
    from utils import separar_colunas
    metadados, variaveis = separar_colunas(df)

    # Verifica se a métrica e modelo especificados na função estão dentro das listas
    if metrica not in lista_metricas:
        raise ValueError("metrica deve conter uma das seguinte 4 métricas: 'RMSE', 'MAE', 'bias', 'r2'")
    elif modelo not in lista_modelos:
        raise ValueError("modelo deve conter um dos seguinte 3 modelos: 'media', 'mediana', 'KNN'")
    else:
        pass

    if modelo == 'media':
        from imputacao import mean_imput
        df_artificial_parcial, df_artificial_total = criar_validacao_cruzada(df, pct_remover=0.05, metadados=metadados)
        df_artificial_parcial = mean_imput(df_artificial_parcial, metadados=metadados, variaveis=variaveis, return_reduced=True)
        df_artificial_total = mean_imput(df_artificial_total, metadados=metadados, variaveis=variaveis, return_reduced=True)

    elif modelo == 'mediana':
        from imputacao import median_imput
        df_artificial_parcial, df_artificial_total = criar_validacao_cruzada(df, pct_remover=0.05, metadados=metadados)
        df_artificial_parcial = median_imput(df_artificial_parcial, metadados=metadados, variaveis=variaveis, return_reduced=True)
        df_artificial_total = median_imput(df_artificial_total, metadados=metadados, variaveis=variaveis, return_reduced=True)

    elif modelo.lower() == 'knn':
        from imputacao import knn_imput
        df_artificial_parcial, df_artificial_total = criar_validacao_cruzada(df, pct_remover=0.15, metadados=metadados)
        df_artificial_parcial = knn_imput(df_artificial_parcial, metadados=metadados, variaveis=variaveis, return_reduced=True)
        df_artificial_total = knn_imput(df_artificial_total, metadados=metadados, variaveis=variaveis, return_reduced=True)

    if metrica.lower() == 'rmse':
        from sklearn.metrics import root_mean_squared_error

        df_verd, df_art, colunas =  preparar_para_metricas(df, df_artificial_parcial, metadados)

        return root_mean_squared_error(df_verd, df_art)
    
    elif metrica.lower() == 'mae':
        from sklearn.metrics import mean_absolute_error

        df_verd, df_art, colunas =  preparar_para_metricas(df, df_artificial_parcial, metadados)

        return mean_absolute_error(df_verd, df_art)
    
    elif metrica == 'r2':
        from sklearn.metrics import r2_score

        df_verd, df_art, colunas =  preparar_para_metricas(df, df_artificial_parcial, metadados)

        return r2_score(df_verd, df_art)

In [5]:
# Teste
df_nan_5, dados_sim_a, dados_sim_b, cols_nan = criar_validacao_cruzada(
    dados_limpos, 
    pct_remover=0.05,  # Remove 5% --> Imputação via média e mediana
    metadados=metadados
)

df_nan_15, dados_sim_a, dados_sim_b, cols_nan = criar_validacao_cruzada(
    dados_limpos, 
    pct_remover=0.15,  # Remove 15% --> Imputação via KNN
    metadados=metadados
)

from imputacao import mean_imput, median_imput, knn_imput

# Apenas colunas com NaN originalmente
imput_mean_a = mean_imput(dados_sim_a, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_median_a = median_imput(dados_sim_a, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_knn_a = knn_imput(dados_sim_a, metadados=metadados, variaveis=variaveis, return_reduced=True)

# DataFrame completo
imput_mean_b = mean_imput(dados_sim_b, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_median_b = median_imput(dados_sim_b, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_knn_b = knn_imput(dados_sim_b, metadados=metadados, variaveis=variaveis, return_reduced=True)

Dataset completo: (377, 20)
Colunas com NaN original: ['COR', 'TURB.', 'pH', 'ALC.', 'AC.', 'O.C.', 'O.D.', 'Cl', 'DUR.', 'Fe', 'Mn', 'Cond.', 'Cianobacteria', 'C.F.', 'Clorofila', 'F']
Valores removidos simulados: 5.0% (18 por coluna)
Dataset completo: (377, 20)
Colunas com NaN original: ['COR', 'TURB.', 'pH', 'ALC.', 'AC.', 'O.C.', 'O.D.', 'Cl', 'DUR.', 'Fe', 'Mn', 'Cond.', 'Cianobacteria', 'C.F.', 'Clorofila', 'F']
Valores removidos simulados: 15.0% (56 por coluna)
Valores faltantes após imputação KNN:
ALC.             0
AC.              0
Cianobacteria    0
Clorofila        0
dtype: int64
Valores faltantes após imputação KNN:
COR              0
TURB.            0
pH               0
ALC.             0
AC.              0
O.C.             0
O.D.             0
Cl               0
DUR.             0
Fe               0
Mn               0
Cond.            0
Cianobacteria    0
C.F.             0
Clorofila        0
F                0
dtype: int64
Valores faltantes após imputação KNN:
COR    

# RMSE

In [ ]:
from sklearn.metrics import root_mean_squared_error

# Cenário A: Imputa apenas nas colunas que tinham NaN
ref_a_prep, imput_mean_a_prep, _ = preparar_para_metricas(ref_a, imput_mean_a, metadados)
ref_a_prep, imput_median_a_prep, _ = preparar_para_metricas(ref_a, imput_median_a, metadados)
ref_a_prep, imput_knn_a_prep, _ = preparar_para_metricas(ref_a, imput_knn_a, metadados)

rmse_mean_a = root_mean_squared_error(ref_a_prep, imput_mean_a_prep)
rmse_median_a = root_mean_squared_error(ref_a_prep, imput_median_a_prep)
rmse_knn_a = root_mean_squared_error(ref_a_prep, imput_knn_a_prep)

print("Cenário A - RMSE (apenas colunas com NaN original):")
print(f"  Mean:   {rmse_mean_a:.6f}")
print(f"  Median: {rmse_median_a:.6f}")
print(f"  KNN:    {rmse_knn_a:.6f}")

# Cenário B: Imputa em todas as colunas
ref_b_prep, imput_mean_b_prep, _ = preparar_para_metricas(ref_b, imput_mean_b, metadados)
ref_b_prep, imput_median_b_prep, _ = preparar_para_metricas(ref_b, imput_median_b, metadados)
ref_b_prep, imput_knn_b_prep, _ = preparar_para_metricas(ref_b, imput_knn_b, metadados)

rmse_mean_b = root_mean_squared_error(ref_b_prep, imput_mean_b_prep)
rmse_median_b = root_mean_squared_error(ref_b_prep, imput_median_b_prep)
rmse_knn_b = root_mean_squared_error(ref_b_prep, imput_knn_b_prep)

print("\nCenário B - RMSE (todas as colunas):")
print(f"  Mean:   {rmse_mean_b:.6f}")
print(f"  Median: {rmse_median_b:.6f}")
print(f"  KNN:    {rmse_knn_b:.6f}")

In [ ]:
from sklearn.metrics import root_mean_squared_error

# Preparar dados para métricas
df_verd, df_pred, colunas = preparar_para_metricas(dados_limpos, imput_mean_limpos, metadados)

# Calcular RMSE
rmse = root_mean_squared_error(df_verd, df_pred)
print(f"RMSE (Mean Imputation): {rmse:.6f}")
rmse

rmse_mean_a = rmse(ref_a, imput_mean_a)

Colunas usadas para métricas: ['COR', 'Fe', 'Mn', 'O.D.', 'pH', 'TURB.', 'Cond.', 'O.C.', 'Cl', 'DUR.']
Total de colunas: 10


ValueError: Input contains NaN.

In [13]:
# Diagnóstico - ver quantos NaN em cada dataframe
print("NaN em df_verd:")
print(df_verd.isna().sum())
print("\nNaN em df_pred:")
print(df_pred.isna().sum())

NaN em df_verd:
COR       3
Fe       21
Mn       21
O.D.     16
pH        3
TURB.     3
Cond.     9
O.C.     21
Cl       21
DUR.     21
dtype: int64

NaN em df_pred:
COR      0
Fe       0
Mn       0
O.D.     0
pH       0
TURB.    0
Cond.    0
O.C.     0
Cl       0
DUR.     0
dtype: int64


# MAE

In [ ]:
from sklearn.metrics import mean_absolute_error

# Cenário A
mae_mean_a = mean_absolute_error(ref_a_prep, imput_mean_a_prep)
mae_median_a = mean_absolute_error(ref_a_prep, imput_median_a_prep)
mae_knn_a = mean_absolute_error(ref_a_prep, imput_knn_a_prep)

print("Cenário A - MAE (apenas colunas com NaN original):")
print(f"  Mean:   {mae_mean_a:.6f}")
print(f"  Median: {mae_median_a:.6f}")
print(f"  KNN:    {mae_knn_a:.6f}")

# Cenário B
mae_mean_b = mean_absolute_error(ref_b_prep, imput_mean_b_prep)
mae_median_b = mean_absolute_error(ref_b_prep, imput_median_b_prep)
mae_knn_b = mean_absolute_error(ref_b_prep, imput_knn_b_prep)

print("\nCenário B - MAE (todas as colunas):")
print(f"  Mean:   {mae_mean_b:.6f}")
print(f"  Median: {mae_median_b:.6f}")
print(f"  KNN:    {mae_knn_b:.6f}")

In [ ]:
from sklearn.metrics import mean_absolute_error

# Preparar dados para métricas
df_verd, df_pred, colunas = preparar_para_metricas(dados_limpos, imput_mean_limpos, metadados)

# Calcular MAE
mae = mean_absolute_error(df_verd, df_pred)
print(f"MAE (Mean Imputation): {mae:.6f}")
mae

# Bias

In [ ]:
from sklearn.metrics import 

# r²

In [ ]:
from sklearn.metrics import r2_score

# Preparar dados para métricas
df_verd, df_pred, colunas = preparar_para_metricas(dados_limpos, imput_mean_limpos, metadados)

# Calcular R²
r2 = r2_score(df_verd, df_pred)
print(f"R² (Mean Imputation): {r2:.6f}")
r2